# 3. Synthetic photometry and zero points

This notebook computes magnitudes of model spectra in the AB, ST and Vega systems, checks them against NJM's zero points, and shows what the detector type (photon- or energy-counting) changes.

Spectra from a grid are **surface fluxes** (erg s⁻¹ cm⁻² Å⁻¹ at the stellar surface). Magnitudes of a surface flux are very negative (the Sun's V is about −38.4). To get what an observer sees, scale the flux by (R/d)² (section 4).

In [ ]:
import dataclasses

import matplotlib.pyplot as plt
import numpy as np

from sed_tools.api import SED, Filters
from sed_tools.photometry import band_average_flux_lambda, load_filter_curve, load_spectrum

plt.rcParams["figure.dpi"] = 80
SIGMA_SB = 5.670374419e-5  # erg s^-1 cm^-2 K^-4
KURUCZ_CUT = dict(teff_min=4000, teff_max=8000, logg_min=3.0, logg_max=5.0,
                  metallicity_min=-1.0, metallicity_max=0.5)

SED.fetch("Kurucz2003all", **KURUCZ_CUT)
SED.fetch("bbody")
johnson = Filters.fetch("Generic", "Johnson")
twomass = Filters.fetch("2MASS", "2MASS")
vega_path = Filters.fetch_vega()
# NJM's vega_flam.csv is in Å; saying so once avoids a warning at every use.
vega_spectrum = load_spectrum(str(vega_path), wavelength_unit="angstrom")

## 1. A solar spectrum

One variant of the split grid loads by its folder and takes Teff, log g and [M/H]. The integrated surface flux of a model atmosphere is σT⁴.

In [ ]:
sed = SED.local("Kurucz2003all/Kurucz2003all_alpha_0p00_lh_1p25_vtur_2_km_s")
sun = sed(teff=5777, logg=4.44, metallicity=0.0)
ratio = np.trapezoid(sun.flux, sun.wavelength) / (SIGMA_SB * 5777**4)
print(f"∫F dλ / σT⁴ = {ratio:.6f}")
assert abs(ratio - 1) < 1e-3

blackbody = SED.local("bbody")(teff=5777, logg=0.0, metallicity=0.0)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.loglog(sun.wavelength, sun.flux, lw=0.7, label="Kurucz 5777 K, log g 4.44")
ax.loglog(blackbody.wavelength, blackbody.flux, lw=0.7, label="blackbody 5777 K (πB)")
ax.set(xlim=(1000, 1e5), ylim=(1e3, 2e7), xlabel="Wavelength (Å)", ylabel="F$_\\lambda$ (erg/s/cm²/Å)")
ax.legend();

## 2. AB and Vega magnitudes

`photometry(...)` takes band names, curve files or a set's folder name (`"Johnson"`), and `system="AB"` or `"Vega"`. Vega magnitudes need a Vega spectrum: NJM's, from `Filters.fetch_vega()`.
- **AB:** m = −2.5 log₁₀(⟨f_ν⟩ / 3631 Jy).
- **Vega:** m = −2.5 log₁₀(⟨f_λ⟩ / ⟨f_λ,Vega⟩).

Each band average uses the curve's own detector weighting. Results are keyed by band name, so sets that share a band name (Johnson and 2MASS both have J) are asked for one at a time.

In [ ]:
ab = {"Johnson": sun.photometry(johnson, system="AB"), "2MASS": sun.photometry(twomass, system="AB")}
vega = {"Johnson": sun.photometry(johnson, system="Vega", vega_spectrum=vega_spectrum),
        "2MASS": sun.photometry(twomass, system="Vega", vega_spectrum=vega_spectrum)}
for name in ab:
    for band, result in ab[name].items():
        print(f"{name:8s} {band:3s} AB {result.magnitude:8.3f}   Vega {vega[name][band].magnitude:8.3f}")
ab, vega = {**ab["Johnson"], **ab["2MASS"]}, {**vega["Johnson"], **vega["2MASS"]}  # J below is 2MASS J
b_v = vega["B"].magnitude - vega["V"].magnitude
print(f"B−V (Sun) = {b_v:.3f}")
assert 0.60 < b_v < 0.72

## 3. ST magnitudes, and NJM's zero points

ST magnitudes use a flat f_λ reference: m_ST = −2.5 log₁₀(⟨f_λ⟩ / 3.631×10⁻⁹ erg s⁻¹ cm⁻² Å⁻¹), that is −2.5 log₁₀⟨f_λ⟩ − 21.10. `photometry` gives AB and Vega; ST comes from the band-averaged f_λ.

NJM's `properties.json` gives each band's zero points (`ZeroPointAB_flam`, `ZeroPointST_flam`, `ZeroPointVega_flam`) and the offsets `ABminusVega` and `STminusVega`. These offsets do not depend on the star: m_AB − m_Vega is the same for every spectrum. Our magnitudes reproduce them.

In [ ]:
props = {**Filters.properties("Generic", "Johnson")["bands"], **Filters.properties("2MASS", "2MASS")["bands"]}
print("band   m_ST     m_AB−m_Vega (NJM)      m_ST−m_Vega (NJM)")
for band, folder in (("B", johnson), ("V", johnson), ("J", twomass), ("Ks", twomass)):
    curve = load_filter_curve(folder / f"{band}.dat")
    m_st = -2.5 * np.log10(band_average_flux_lambda(sun.wavelength, sun.flux, curve) / props[band]["ZeroPointST_flam"])
    ab_vega = ab[band].magnitude - vega[band].magnitude
    st_vega = m_st - vega[band].magnitude
    print(f"{band:3s} {m_st:8.3f}   {ab_vega:7.4f} ({props[band]['ABminusVega']:7.4f})"
          f"      {st_vega:7.4f} ({props[band]['STminusVega']:7.4f})")
    assert abs(ab_vega - props[band]["ABminusVega"]) < 2e-3
    assert abs(st_vega - props[band]["STminusVega"]) < 2e-3

2MASS Ks: m_AB − m_Vega ≈ 1.86, the known AB offset of Ks.

## 4. From surface flux to what an observer sees

Scaling by (R/d)² turns the surface flux into the flux at distance d. For the Sun at 1 AU this gives the familiar V ≈ −26.7; at 10 pc it gives the absolute magnitude M_V ≈ 4.8.

In [ ]:
R_SUN, AU, PC = 6.957e10, 1.495978707e13, 3.0856775814913673e18  # cm
for label, distance in (("1 AU", AU), ("10 pc", 10 * PC)):
    seen = dataclasses.replace(sun, flux=sun.flux * (R_SUN / distance) ** 2)
    v = seen.photometry(johnson / "V.dat", system="Vega", vega_spectrum=vega_spectrum)["V"].magnitude
    print(f"V at {label:5s} = {v:7.3f}")
assert abs(vega["V"].magnitude - 5 * np.log10(R_SUN / AU) - (-26.76)) < 0.1

## 5. Photon- vs energy-counting

The same curve gives different magnitudes under the two weightings, `T(λ)` (energy) and `T(λ)·λ` (photon). Johnson's curves are energy-counting. Treating them as photon-counting shifts the Sun's colours by a few hundredths of a magnitude, which is why every curve records its type.

In [ ]:
def magnitude(curve, flux_source, system_zero):
    return -2.5 * np.log10(band_average_flux_lambda(flux_source.wavelength, flux_source.flux, curve) / system_zero)

for detector in ("energy", "photon"):
    mags = {}
    for band in ("B", "V"):
        curve = dataclasses.replace(load_filter_curve(johnson / f"{band}.dat"), detector_type=detector)
        zero = band_average_flux_lambda(vega_spectrum.wavelength, vega_spectrum.flux, curve)
        mags[band] = magnitude(curve, sun, zero)
    print(f"{detector:6s}: B−V = {mags['B'] - mags['V']:.4f}")

## 6. Colours along a sequence

B−V and J−Ks fall as Teff rises, at fixed log g and [M/H].

In [ ]:
teffs = np.arange(4000, 8001, 250)
colours = []
for teff in teffs:
    spectrum = sed(teff=teff, logg=4.5, metallicity=0.0)
    m = spectrum.photometry(johnson, system="Vega", vega_spectrum=vega_spectrum)
    n = spectrum.photometry(twomass, system="Vega", vega_spectrum=vega_spectrum)
    colours.append((m["B"].magnitude - m["V"].magnitude, n["J"].magnitude - n["Ks"].magnitude))
colours = np.array(colours)
assert np.all(np.diff(colours[:, 0]) < 0)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(teffs, colours[:, 0], "o-", label="B−V")
ax.plot(teffs, colours[:, 1], "s-", label="J−Ks")
ax.set(xlabel="Teff (K)", ylabel="colour (Vega mag)", title="log g 4.5, [M/H] 0")
ax.legend();

## Command-line equivalent

The flux cube tool prints the photometry of an interpolated spectrum (it asks for what is not given):

```bash
python -m sed_tools.cube.tool --help
```